In [0]:
use catalog apjtechup;
use database gold;

In [0]:
%python
import requests
import time
import pandas as pd
from datetime import datetime
import json
import re

# Configuration - Update these values
CSV_FILE_PATH = 'benchmark_queries/final.csv'  # Update with your CSV file path
WAREHOUSE_ID = "87e50f8f59e42ed2"  # Update with your warehouse ID
DEFAULT_CATALOG = "apjtechup"  # Default catalog
DEFAULT_SCHEMA = "gold_dimensional"  # Default schema
LOOP_COUNT = 5  # Number of times to loop through and re-submit all queries

# Get Databricks context
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
api_url = ctx.apiUrl().get()
api_token = ctx.apiToken().get()

# API endpoints
submit_endpoint = f"{api_url}/api/2.0/sql/statements"
status_endpoint = f"{api_url}/api/2.0/sql/statements/"
query_history_endpoint = f"{api_url}/api/2.0/sql/history/queries"
warehouse_endpoint = f"{api_url}/api/2.0/sql/warehouses/{WAREHOUSE_ID}"

headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

print(f"Reading queries from: {CSV_FILE_PATH}")
print(f"Using warehouse: {WAREHOUSE_ID}")
print(f"Default catalog: {DEFAULT_CATALOG}")
print(f"Default schema: {DEFAULT_SCHEMA}")
print(f"Loop count: {LOOP_COUNT}")
print(f"API URL: {api_url}")
print("="*80)

# Function to add RAND() IS NOT NULL filter to SQL query
def add_rand_filter(query):
    """
    Adds 'RAND() IS NOT NULL' filter to a SQL query.
    - If query has WHERE clause, adds with AND
    - If no WHERE clause, inserts WHERE before ORDER BY, GROUP BY, HAVING, LIMIT, QUALIFY, etc.
    - Handles window functions (OVER clauses) correctly by not inserting inside parentheses
    """
    query = query.strip()
    if not query:
        return query
    
    # Remove trailing semicolon if present
    has_semicolon = query.endswith(';')
    if has_semicolon:
        query = query[:-1].strip()
    
    # Helper function to find keyword positions at depth 0 (not inside parentheses)
    def find_keyword_at_depth_zero(text, keyword_pattern):
        """Find keyword positions that are not inside parentheses"""
        matches = []
        depth = 0
        
        for match in re.finditer(keyword_pattern, text, re.IGNORECASE):
            # Calculate parentheses depth at this position
            depth = 0
            for i in range(match.start()):
                if text[i] == '(':
                    depth += 1
                elif text[i] == ')':
                    depth -= 1
            
            # Only include matches at depth 0 (not inside parentheses)
            if depth == 0:
                matches.append(match)
        
        return matches
    
    # Pattern to find WHERE clause at depth 0
    where_pattern = r'\bWHERE\b'
    where_matches = find_keyword_at_depth_zero(query, where_pattern)
    
    if where_matches:
        # Query has WHERE clause at main level - need to add AND RAND() IS NOT NULL
        where_match = where_matches[0]  # Use first WHERE at depth 0
        
        # Keywords that come after WHERE clause
        post_where_keywords = [
            r'\bGROUP\s+BY\b',
            r'\bHAVING\b',
            r'\bQUALIFY\b',
            r'\bWINDOW\b',
            r'\bORDER\s+BY\b',
            r'\bLIMIT\b',
            r'\bOFFSET\b',
            r'\bFETCH\b'
        ]
        
        # Find the earliest occurrence of any post-WHERE keyword at depth 0
        where_end = len(query)
        for keyword_pattern in post_where_keywords:
            keyword_matches = find_keyword_at_depth_zero(query[where_match.end():], keyword_pattern)
            if keyword_matches:
                actual_pos = where_match.end() + keyword_matches[0].start()
                where_end = min(where_end, actual_pos)
        
        # Insert AND RAND() IS NOT NULL before the next clause
        modified_query = query[:where_end].rstrip() + ' AND RAND() IS NOT NULL ' + query[where_end:]
    else:
        # Query doesn't have WHERE clause - need to add WHERE RAND() IS NOT NULL
        # Find position to insert (before GROUP BY, HAVING, ORDER BY, LIMIT, etc.)
        
        insert_keywords = [
            r'\bGROUP\s+BY\b',
            r'\bHAVING\b',
            r'\bQUALIFY\b',
            r'\bWINDOW\b',
            r'\bORDER\s+BY\b',
            r'\bLIMIT\b',
            r'\bOFFSET\b',
            r'\bFETCH\b'
        ]
        
        # Find the earliest occurrence of any keyword at depth 0
        insert_pos = len(query)
        for keyword_pattern in insert_keywords:
            keyword_matches = find_keyword_at_depth_zero(query, keyword_pattern)
            if keyword_matches:
                insert_pos = min(insert_pos, keyword_matches[0].start())
        
        # Insert WHERE RAND() IS NOT NULL at the appropriate position
        modified_query = query[:insert_pos].rstrip() + ' WHERE RAND() IS NOT NULL ' + query[insert_pos:]
    
    # Add back semicolon if it was present
    if has_semicolon:
        modified_query += ';'
    
    return modified_query

# Step 1: Read CSV file with SQL queries
with open(CSV_FILE_PATH, 'r') as f:
    queries = [line.strip() for line in f.readlines() if line.strip()]

print(f"\nLoaded {len(queries)} queries from CSV file")
print("\nAdding RAND() IS NOT NULL filter to each query...\n")

# Step 2: Add RAND() IS NOT NULL filter to each query
modified_queries = []
for idx, query in enumerate(queries, 1):
    try:
        modified_query = add_rand_filter(query)
        modified_queries.append(modified_query)
        
        # Show first 5 examples
        if idx <= 5:
            print(f"Query {idx} transformation:")
            print(f"  Original: {query[:150]}..." if len(query) > 150 else f"  Original: {query}")
            print(f"  Modified: {modified_query[:150]}..." if len(modified_query) > 150 else f"  Modified: {modified_query}")
            print()
    except Exception as e:
        print(f"⚠ Error modifying query {idx}: {str(e)}")
        print(f"  Query: {query[:100]}...")
        modified_queries.append(query)  # Use original if modification fails

print(f"Modified {len(modified_queries)} queries\n")
print("="*80)

# Initialize tracking for all loops
all_statement_ids = []
all_submission_results = []
all_statement_details = {}

# Step 3: Submit all queries from all loops without waiting
print(f"\n{'='*80}")
print(f"SUBMITTING ALL QUERIES FROM ALL {LOOP_COUNT} LOOPS")
print(f"{'='*80}\n")

for loop_iteration in range(1, LOOP_COUNT + 1):
    print(f"\nSubmitting queries for Loop {loop_iteration}...\n")
    
    for idx, (original_query, modified_query) in enumerate(zip(queries, modified_queries), 1):
        print(f"[Loop {loop_iteration}] Submitting query {idx}/{len(modified_queries)}...")
        
        payload = {
            "statement": modified_query,
            "wait_timeout": "0s",
            "warehouse_id": WAREHOUSE_ID,
            "catalog": DEFAULT_CATALOG,
            "schema": DEFAULT_SCHEMA,
            "disposition": "EXTERNAL_LINKS",
            "format": "ARROW_STREAM"
        }
        
        try:
            response = requests.post(submit_endpoint, headers=headers, json=payload)
            response.raise_for_status()
            result = response.json()
            statement_id = result.get('statement_id')
            all_statement_ids.append(statement_id)
            
            submission_result = {
                'loop_iteration': loop_iteration,
                'query_number': idx,
                'statement_id': statement_id,
                'query': original_query[:100] + '...' if len(original_query) > 100 else original_query
            }
            all_submission_results.append(submission_result)
            
            # Initialize tracking
            all_statement_details[statement_id] = {
                'loop_iteration': loop_iteration,
                'statement_id': statement_id,
                'status': 'PENDING',
                'start_time': None,
                'end_time': None,
                'duration_ms': None,
                'duration_seconds': None,
                'rows_produced': None,
                'error_message': None
            }
            
            print(f"  ✓ Statement ID: {statement_id}")
        except Exception as e:
            print(f"  ✗ Error submitting query {idx}: {str(e)}")
            all_statement_ids.append(None)
            
            submission_result = {
                'loop_iteration': loop_iteration,
                'query_number': idx,
                'statement_id': None,
                'query': original_query[:100] + '...' if len(original_query) > 100 else original_query,
                'error': str(e)
            }
            all_submission_results.append(submission_result)
    time.sleep(10)

print(f"\n{'='*80}")
print(f"Submitted {len([s for s in all_statement_ids if s])} statements successfully across all loops")
print(f"{'='*80}\n")

# Step 4: Monitor all statement executions together
print(f"Monitoring all statement executions...\n")

while True:
    all_completed = True
    
    for statement_id in [sid for sid in all_statement_ids if sid]:
        if all_statement_details[statement_id]['status'] not in ['SUCCEEDED', 'FAILED', 'CANCELED', 'CLOSED']:
            try:
                response = requests.get(f"{status_endpoint}{statement_id}", headers=headers)
                response.raise_for_status()
                result = response.json()
                
                status_obj = result.get('status', {})
                status = status_obj.get('state', 'UNKNOWN')
                all_statement_details[statement_id]['status'] = status
                
                # Extract error message if failed
                if status == 'FAILED' and 'error' in status_obj:
                    error_info = status_obj['error']
                    error_message = error_info.get('message', 'Unknown error')
                    all_statement_details[statement_id]['error_message'] = error_message
                
                if status not in ['SUCCEEDED', 'FAILED', 'CANCELED', 'CLOSED']:
                    all_completed = False
                    
            except Exception as e:
                print(f"Error checking status for {statement_id}: {str(e)}")
                all_completed = False
    
    # Print status summary
    status_counts = {}
    for details in all_statement_details.values():
        status = details['status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Status: {dict(status_counts)}")
    
    if all_completed:
        print(f"\n✓ All statements completed!\n")
        break
    
    print("  Waiting 10 seconds before next check...")
    time.sleep(10)

# Step 5: Fetch timing information from Query History API
print(f"{'='*80}")
print(f"Fetching timing information from Query History API...")
print(f"{'='*80}\n")

for statement_id in [sid for sid in all_statement_ids if sid]:
    try:
        # Query History API to get timing details
        history_response = requests.get(
            query_history_endpoint,
            headers=headers,
            json={"filter_by": {"statement_ids": [statement_id]}}
        )
        
        if history_response.status_code == 200:
            history_result = history_response.json()
            if history_result.get('res') and len(history_result['res']) > 0:
                query_info = history_result['res'][0]
                
                # Extract timing information
                start_time_ms = query_info.get('query_start_time_ms')
                end_time_ms = query_info.get('query_end_time_ms')
                duration_ms = query_info.get('duration')
                rows_produced = query_info.get('rows_produced')
                
                if start_time_ms:
                    start_time = datetime.fromtimestamp(start_time_ms / 1000).isoformat()
                    all_statement_details[statement_id]['start_time'] = start_time
                if end_time_ms:
                    end_time = datetime.fromtimestamp(end_time_ms / 1000).isoformat()
                    all_statement_details[statement_id]['end_time'] = end_time
                if duration_ms:
                    all_statement_details[statement_id]['duration_ms'] = duration_ms
                    all_statement_details[statement_id]['duration_seconds'] = duration_ms / 1000.0
                if rows_produced is not None:
                    all_statement_details[statement_id]['rows_produced'] = rows_produced
                
                print(f"✓ Fetched timing for {statement_id}: {duration_ms}ms, {rows_produced} rows")
            else:
                print(f"⚠ No history found for {statement_id}")
        else:
            print(f"✗ Error fetching history for {statement_id}: {history_response.status_code}")
            
    except Exception as e:
        print(f"✗ Exception fetching history for {statement_id}: {str(e)}")

print(f"\n{'='*80}")
print(f"Creating summary dataframe...")
print(f"{'='*80}\n")

# Step 6: Create summary dataframe with all loops
summary_data = []
for submission in all_submission_results:
    statement_id = submission['statement_id']
    if statement_id and statement_id in all_statement_details:
        details = all_statement_details[statement_id]
        summary_data.append({
            'loop_iteration': submission['loop_iteration'],
            'query_number': submission['query_number'],
            'statement_id': statement_id,
            'status': details['status'],
            'start_time': details['start_time'],
            'end_time': details['end_time'],
            'duration_ms': details['duration_ms'],
            'duration_seconds': details['duration_seconds'],
            'rows_produced': details['rows_produced'],
            'error_message': details['error_message'],
            'query_preview': submission['query']
        })
    else:
        summary_data.append({
            'loop_iteration': submission['loop_iteration'],
            'query_number': submission['query_number'],
            'statement_id': statement_id,
            'status': 'SUBMISSION_FAILED',
            'start_time': None,
            'end_time': None,
            'duration_ms': None,
            'duration_seconds': None,
            'rows_produced': None,
            'error_message': submission.get('error', 'Failed to submit'),
            'query_preview': submission['query']
        })

summary_df = pd.DataFrame(summary_data)

print("Summary DataFrame (all loops):")
print(summary_df.to_string())
print(f"\n{'='*80}")
print(f"Total executions: {len(summary_df)}")
print(f"Total loops: {LOOP_COUNT}")
print(f"Queries per loop: {len(queries)}")
print(f"Succeeded: {len(summary_df[summary_df['status'] == 'SUCCEEDED'])}")
print(f"Failed: {len(summary_df[summary_df['status'].isin(['FAILED', 'SUBMISSION_FAILED'])])}")

# Calculate min start time and max end time across all loops
if summary_df['start_time'].notna().any() and summary_df['end_time'].notna().any():
    # Convert to datetime for calculation
    start_times = pd.to_datetime(summary_df['start_time'].dropna(), errors='coerce')
    end_times = pd.to_datetime(summary_df['end_time'].dropna(), errors='coerce')
    
    min_start_time = start_times.min()
    max_end_time = end_times.max()
    total_elapsed_time = (max_end_time - min_start_time).total_seconds()
    
    print(f"\nOverall Timing Summary (all loops):")
    print(f"  Minimum start time: {min_start_time}")
    print(f"  Maximum end time: {max_end_time}")
    print(f"  Total elapsed time (start to end): {total_elapsed_time:.2f} seconds ({total_elapsed_time/60:.2f} minutes)")

if summary_df['duration_seconds'].notna().any():
    print(f"\nQuery Duration Statistics (all loops):")
    print(f"  Average duration: {summary_df['duration_seconds'].mean():.2f} seconds")
    print(f"  Min duration: {summary_df['duration_seconds'].min():.2f} seconds")
    print(f"  Max duration: {summary_df['duration_seconds'].max():.2f} seconds")
    print(f"  Total duration (sum): {summary_df['duration_seconds'].sum():.2f} seconds")
    
if summary_df['rows_produced'].notna().any():
    print(f"\nRows Produced Statistics (all loops):")
    print(f"  Total rows produced: {summary_df['rows_produced'].sum():.0f}")
    print(f"  Average rows per query: {summary_df['rows_produced'].mean():.0f}")

# Per-loop statistics
print(f"\n{'='*80}")
print("Per-Loop Statistics:")
print(f"{'='*80}")
for loop_num in range(1, LOOP_COUNT + 1):
    loop_df = summary_df[summary_df['loop_iteration'] == loop_num]
    if len(loop_df) > 0 and loop_df['duration_seconds'].notna().any():
        print(f"\nLoop {loop_num}:")
        print(f"  Queries executed: {len(loop_df)}")
        print(f"  Succeeded: {len(loop_df[loop_df['status'] == 'SUCCEEDED'])}")
        print(f"  Failed: {len(loop_df[loop_df['status'].isin(['FAILED', 'SUBMISSION_FAILED'])])}")
        print(f"  Average duration: {loop_df['duration_seconds'].mean():.2f} seconds")
        print(f"  Total duration: {loop_df['duration_seconds'].sum():.2f} seconds")
        
        if loop_df['start_time'].notna().any() and loop_df['end_time'].notna().any():
            loop_start = pd.to_datetime(loop_df['start_time'].dropna(), errors='coerce').min()
            loop_end = pd.to_datetime(loop_df['end_time'].dropna(), errors='coerce').max()
            loop_elapsed = (loop_end - loop_start).total_seconds()
            print(f"  Loop elapsed time: {loop_elapsed:.2f} seconds ({loop_elapsed/60:.2f} minutes)")
    
print(f"\n{'='*80}")

# Display as Spark DataFrame for better visualization
spark_df = spark.createDataFrame(summary_df)
display(spark_df)

# Step 7: Shutdown SQL Warehouse
print(f"\n{'='*80}")
print(f"SHUTTING DOWN SQL WAREHOUSE")
print(f"{'='*80}\n")

try:
    print(f"Stopping SQL Warehouse: {WAREHOUSE_ID}...")
    stop_payload = {}
    stop_response = requests.post(f"{warehouse_endpoint}/stop", headers=headers, json=stop_payload)
    
    if stop_response.status_code == 200:
        print(f"✓ SQL Warehouse {WAREHOUSE_ID} stop command sent successfully")
        print(f"  The warehouse will shut down after completing all running queries.")
    else:
        print(f"⚠ Warning: Failed to stop warehouse. Status code: {stop_response.status_code}")
        print(f"  Response: {stop_response.text}")
except Exception as e:
    print(f"✗ Error stopping SQL Warehouse: {str(e)}")

print(f"\n{'='*80}")
print(f"BENCHMARK COMPLETE")
print(f"{'='*80}")

In [0]:
%python
display(summary_df)

In [0]:
%python

# Calculate min start time and max end time across all loops
if summary_df['start_time'].notna().any() and summary_df['end_time'].notna().any():
    # Convert to datetime for calculation
    start_times = pd.to_datetime(summary_df['start_time'].dropna(), errors='coerce')
    end_times = pd.to_datetime(summary_df['end_time'].dropna(), errors='coerce')
    
    min_start_time = start_times.min()
    max_end_time = end_times.max()
    total_elapsed_time = (max_end_time - min_start_time).total_seconds()
    
    print(f"\nOverall Timing Summary (all loops):")
    print(f"  Minimum start time: {min_start_time}")
    print(f"  Maximum end time: {max_end_time}")
    print(f"  Total elapsed time (start to end): {total_elapsed_time:.2f} seconds ({total_elapsed_time/60:.2f} minutes)")

if summary_df['duration_seconds'].notna().any():
    print(f"\nQuery Duration Statistics (all loops):")
    print(f"  Average duration: {summary_df['duration_seconds'].mean():.2f} seconds")
    print(f"  Min duration: {summary_df['duration_seconds'].min():.2f} seconds")
    print(f"  Max duration: {summary_df['duration_seconds'].max():.2f} seconds")
    print(f"  Total duration (sum): {summary_df['duration_seconds'].sum():.2f} seconds")
    
if summary_df['rows_produced'].notna().any():
    print(f"\nRows Produced Statistics (all loops):")
    print(f"  Total rows produced: {summary_df['rows_produced'].sum():.0f}")
    print(f"  Average rows per query: {summary_df['rows_produced'].mean():.0f}")

# Per-loop statistics
print(f"\n{'='*80}")
print("Per-Loop Statistics:")
print(f"{'='*80}")
for loop_num in range(1, LOOP_COUNT + 1):
    loop_df = summary_df[summary_df['loop_iteration'] == loop_num]
    if len(loop_df) > 0 and loop_df['duration_seconds'].notna().any():
        print(f"\nLoop {loop_num}:")
        print(f"  Queries executed: {len(loop_df)}")
        print(f"  Succeeded: {len(loop_df[loop_df['status'] == 'SUCCEEDED'])}")
        print(f"  Failed: {len(loop_df[loop_df['status'].isin(['FAILED', 'SUBMISSION_FAILED'])])}")
        print(f"  Average duration: {loop_df['duration_seconds'].mean():.2f} seconds")
        print(f"  Total duration: {loop_df['duration_seconds'].sum():.2f} seconds")
        
        if loop_df['start_time'].notna().any() and loop_df['end_time'].notna().any():
            loop_start = pd.to_datetime(loop_df['start_time'].dropna()).min()
            loop_end = pd.to_datetime(loop_df['end_time'].dropna(), format='ISO8601').max()
            loop_elapsed = (loop_end - loop_start).total_seconds()
            print(f"  Loop elapsed time: {loop_elapsed:.2f} seconds ({loop_elapsed/60:.2f} minutes)")
    
print(f"\n{'='*80}")

# Display as Spark DataFrame for better visualization
spark_df = spark.createDataFrame(summary_df)
display(spark_df)

# Step 7: Shutdown SQL Warehouse
print(f"\n{'='*80}")
print(f"SHUTTING DOWN SQL WAREHOUSE")
print(f"{'='*80}\n")

try:
    print(f"Stopping SQL Warehouse: {WAREHOUSE_ID}...")
    stop_payload = {}
    stop_response = requests.post(f"{warehouse_endpoint}/stop", headers=headers, json=stop_payload)
    
    if stop_response.status_code == 200:
        print(f"✓ SQL Warehouse {WAREHOUSE_ID} stop command sent successfully")
        print(f"  The warehouse will shut down after completing all running queries.")
    else:
        print(f"⚠ Warning: Failed to stop warehouse. Status code: {stop_response.status_code}")
        print(f"  Response: {stop_response.text}")
except Exception as e:
    print(f"✗ Error stopping SQL Warehouse: {str(e)}")

print(f"\n{'='*80}")
print(f"BENCHMARK COMPLETE")
print(f"{'='*80}")